# Supervised Learning Lab

Hands-on implementation of core supervised learning algorithms.

**Topics:**
- Linear Regression from scratch
- Logistic Regression
- Decision Trees
- Model Comparison

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_regression, load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")

## 1. Linear Regression from Scratch

Linear regression finds weights $ and bias $ to minimize:
38967\mathcal{L} = rac{1}{n}\sum_{i=1}^n (y_i - (w^T x_i + b))^238967

We will implement gradient descent to learn these parameters.

In [ ]:
# Generate data
np.random.seed(42)
X = 2 * np.random.rand(100, 1)
y = 4 + 3 * X[:, 0] + np.random.randn(100) * 0.5

plt.scatter(X, y, alpha=0.6)
plt.xlabel("X"); plt.ylabel("y")
plt.title("Linear Regression Data")
plt.show()

In [ ]:
class LinearRegressionScratch:
    def __init__(self, lr=0.01, n_iters=1000):
        self.lr = lr
        self.n_iters = n_iters
        self.weights = None
        self.bias = None
        self.losses = []
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        for _ in range(self.n_iters):
            y_pred = X @ self.weights + self.bias
            
            # Gradients
            dw = (2/n_samples) * X.T @ (y_pred - y)
            db = (2/n_samples) * np.sum(y_pred - y)
            
            # Update
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
            # Track loss
            loss = np.mean((y_pred - y)**2)
            self.losses.append(loss)
    
    def predict(self, X):
        return X @ self.weights + self.bias

In [ ]:
# Train our model
model = LinearRegressionScratch(lr=0.1, n_iters=100)
model.fit(X, y)

print(f"Learned: y = {model.weights[0]:.4f} * x + {model.bias:.4f}")
print(f"True:    y = 3.0000 * x + 4.0000")
print(f"Final MSE: {model.losses[-1]:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.scatter(X, y, alpha=0.5)
X_line = np.linspace(0, 2, 100).reshape(-1, 1)
ax1.plot(X_line, model.predict(X_line), "r-", linewidth=2, label="Prediction")
ax1.set_title("Linear Regression Fit"); ax1.legend()

ax2.plot(model.losses)
ax2.set_xlabel("Iteration"); ax2.set_ylabel("MSE")
ax2.set_title("Training Loss")
plt.tight_layout()
plt.show()

### Analytical Solution (Normal Equation)

38967w = (X^TX)^{-1}X^Ty38967

In [ ]:
# Add bias column
X_b = np.c_[np.ones((100, 1)), X]
theta = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
print(f"Normal equation: y = {theta[1]:.4f} * x + {theta[0]:.4f}")
print(f"Gradient descent: y = {model.weights[0]:.4f} * x + {model.bias:.4f}")

## 2. Logistic Regression from Scratch

For binary classification, we use the sigmoid function:
38967\sigma(z) = rac{1}{1 + e^{-z}}38967

Loss function (Binary Cross-Entropy):
38967\mathcal{L} = -rac{1}{n}\sum[y\log(\hat{y}) + (1-y)\log(1-\hat{y})]38967

In [ ]:
class LogisticRegressionScratch:
    def __init__(self, lr=0.01, n_iters=1000):
        self.lr = lr
        self.n_iters = n_iters
        self.losses = []
    
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        for _ in range(self.n_iters):
            z = X @ self.weights + self.bias
            y_pred = self.sigmoid(z)
            
            dw = (1/n_samples) * X.T @ (y_pred - y)
            db = (1/n_samples) * np.sum(y_pred - y)
            
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
            # BCE loss
            loss = -np.mean(y*np.log(y_pred+1e-8) + (1-y)*np.log(1-y_pred+1e-8))
            self.losses.append(loss)
    
    def predict_proba(self, X):
        return self.sigmoid(X @ self.weights + self.bias)
    
    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

In [ ]:
# Generate classification data
X_cls, y_cls = make_classification(n_samples=300, n_features=2, n_redundant=0,
                                     n_informative=2, random_state=42, n_clusters_per_class=1)
scaler = StandardScaler()
X_cls = scaler.fit_transform(X_cls)

X_train, X_test, y_train, y_test = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42)

# Train
log_reg = LogisticRegressionScratch(lr=0.1, n_iters=200)
log_reg.fit(X_train, y_train)

y_pred = log_reg.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

In [ ]:
# Decision boundary visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Decision boundary
xx, yy = np.meshgrid(np.linspace(X_cls[:,0].min()-1, X_cls[:,0].max()+1, 200),
                      np.linspace(X_cls[:,1].min()-1, X_cls[:,1].max()+1, 200))
Z = log_reg.predict_proba(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

ax1.contourf(xx, yy, Z, levels=50, cmap="RdBu", alpha=0.6)
ax1.scatter(X_cls[:,0], X_cls[:,1], c=y_cls, cmap="RdBu", edgecolors="k", s=20)
ax1.set_title("Logistic Regression Decision Boundary")

ax2.plot(log_reg.losses)
ax2.set_xlabel("Iteration"); ax2.set_ylabel("BCE Loss")
ax2.set_title("Training Loss")
plt.tight_layout()
plt.show()

## 3. Decision Trees

Decision trees split data based on feature thresholds to minimize impurity (Gini or Entropy).

**Gini Impurity:**  = 1 - \sum p_i^2$

**Information Gain:**  = H(parent) - \sumrac{n_j}{n}H(child_j)$

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

# Use iris dataset for visualization
iris = load_iris()
X_iris = iris.data[:, [2, 3]]  # petal length, petal width
y_iris = iris.target

# Train decision tree
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_iris, y_iris)

print("Decision Tree Rules:")
print(export_text(dt, feature_names=["petal_length", "petal_width"]))

In [ ]:
# Visualize decision regions
xx, yy = np.meshgrid(np.linspace(X_iris[:,0].min()-0.5, X_iris[:,0].max()+0.5, 300),
                      np.linspace(X_iris[:,1].min()-0.5, X_iris[:,1].max()+0.5, 300))
Z = dt.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.4, cmap="Set1")
scatter = plt.scatter(X_iris[:,0], X_iris[:,1], c=y_iris, cmap="Set1", edgecolors="k", s=40)
plt.xlabel("Petal Length"); plt.ylabel("Petal Width")
plt.title(f"Decision Tree (max_depth=3) - Accuracy: {dt.score(X_iris, y_iris):.2%}")
plt.colorbar(scatter)
plt.show()

In [ ]:
# Effect of max_depth on overfitting
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, depth in zip(axes, [1, 2, 4, None]):
    dt_temp = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt_temp.fit(X_iris, y_iris)
    Z = dt_temp.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.4, cmap="Set1")
    ax.scatter(X_iris[:,0], X_iris[:,1], c=y_iris, cmap="Set1", edgecolors="k", s=15)
    ax.set_title(f"depth={depth}
acc={dt_temp.score(X_iris, y_iris):.2%}")
plt.suptitle("Decision Tree: Effect of Max Depth")
plt.tight_layout()
plt.show()

## 4. Model Comparison

Let us compare multiple algorithms on the same dataset.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Generate a more complex dataset
X_cmp, y_cmp = make_classification(n_samples=500, n_features=10, n_informative=5,
                                     n_redundant=2, random_state=42)
X_cmp = StandardScaler().fit_transform(X_cmp)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(max_depth=5),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM (RBF)": SVC(kernel="rbf"),
}

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_cmp, y_cmp, cv=5, scoring="accuracy")
    results[name] = scores
    print(f"{name:25s}: {scores.mean():.4f} ± {scores.std():.4f}")

In [ ]:
# Box plot comparison
plt.figure(figsize=(10, 5))
plt.boxplot(results.values(), labels=results.keys())
plt.ylabel("Accuracy")
plt.title("Model Comparison (5-Fold CV)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize decision boundaries for 2D slice
X_2d, y_2d = make_classification(n_samples=300, n_features=2, n_redundant=0,
                                   n_informative=2, random_state=42)
X_2d = StandardScaler().fit_transform(X_2d)

fig, axes = plt.subplots(1, 5, figsize=(20, 3.5))
xx, yy = np.meshgrid(np.linspace(X_2d[:,0].min()-1, X_2d[:,0].max()+1, 200),
                      np.linspace(X_2d[:,1].min()-1, X_2d[:,1].max()+1, 200))

models_2d = {
    "Logistic": LogisticRegression(),
    "KNN": KNeighborsClassifier(5),
    "Tree": DecisionTreeClassifier(max_depth=4),
    "Forest": RandomForestClassifier(50, random_state=42),
    "SVM": SVC(kernel="rbf"),
}

for ax, (name, model) in zip(axes, models_2d.items()):
    model.fit(X_2d, y_2d)
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.4, cmap="RdBu")
    ax.scatter(X_2d[:,0], X_2d[:,1], c=y_2d, cmap="RdBu", edgecolors="k", s=15)
    ax.set_title(f"{name}
acc={model.score(X_2d, y_2d):.2%}")

plt.suptitle("Decision Boundaries Comparison")
plt.tight_layout()
plt.show()

## Key Takeaways

| Algorithm | Pros | Cons |
|-----------|------|------|
| Linear/Logistic Regression | Fast, interpretable | Linear boundaries only |
| KNN | Simple, no training | Slow at prediction, curse of dimensionality |
| Decision Tree | Interpretable, handles non-linearity | Overfits easily |
| Random Forest | Robust, handles overfitting | Less interpretable |
| SVM | Powerful with kernels | Slow on large datasets |

**Next steps:** Try these on real datasets (housing prices, titanic, etc.)!